Agentic AI Analytics Bot v1.4
Changes:


*   Added interactive chat-like capability
*   Added auto-switching between fast (depth = 0) and deep (depth = 1) modes.
*   Added naturally-sounded acknowledgement, for follow up / additional user inputs

Results: works flawelessly.


In [2]:
# Config

# !pip install anthropic -q
# !pip install duckdb -q
import os
import random
import time
import json
import duckdb
import numpy as np
import pandas as pd
import glob
from google.colab import userdata
from google.colab import files


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 13.7 MB/s eta 0:00:00


In [3]:
with open("/content/data_model.json", "r") as f:
    data_model = json.load(f)

SCHEMA = json.dumps(data_model, indent=2)
print(f"Schema loaded: {len(data_model['tables'])} tables")

Schema loaded: 7 tables


In [4]:
# Load a real ecommerce dataset from a public source

# Brazilian ecommerce dataset (real anonymized data, ~100k orders) - Facts
orders = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_orders_dataset.csv")
products = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_products_dataset.csv")
reviews = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_reviews_dataset.csv")

print(f"Orders: {len(orders)} rows")
print(f"Products: {len(products)} rows")
print(f"Reviews: {len(reviews)} rows")
print("\nOrders columns:", list(orders.columns))

# Brazilian ecommerce dataset - Dimmensions
customers = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_customers_dataset.csv")
order_items = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_items_dataset.csv")
payments = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_order_payments_dataset.csv")
sellers_dim = pd.read_csv("https://raw.githubusercontent.com/olist/work-at-olist-data/master/datasets/olist_sellers_dataset.csv")

print(f"Customers: {len(customers)} rows — {list(customers.columns)}")
print(f"Order items: {len(order_items)} rows — {list(order_items.columns)}")
print(f"Payments: {len(payments)} rows — {list(payments.columns)}")
print(f"Sellers: {len(sellers_dim)} rows — {list(sellers_dim.columns)}")


Orders: 99441 rows
Products: 32951 rows
Reviews: 99224 rows

Orders columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']
Customers: 99441 rows — ['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']
Order items: 112650 rows — ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']
Payments: 103886 rows — ['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']
Sellers: 3095 rows — ['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


In [5]:
# Create a data dictionary the agent can reference
tables = {
    "orders": orders,
    "order_items": order_items,
    "products": products,
    "customers": customers,
    "payments": payments,
    "sellers": sellers_dim,
    "reviews": reviews
}

# Load rich schema from JSON file
import json

with open("/content/data_model.json", "r") as f:
    data_model = json.load(f)

SCHEMA = json.dumps(data_model, indent=2)
print(f"Schema loaded: {len(data_model['tables'])} tables")
print(f"Schema length: {len(SCHEMA)} characters")

Schema loaded: 7 tables
Schema length: 19292 characters


In [6]:
from anthropic import Anthropic
os.environ["Claude"] = userdata.get("Claude")
claude_client = Anthropic(api_key=os.environ["Claude"])

# Test it
response = claude_client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2048,
    timeout=60,
    messages=[{"role": "user", "content": "Say hello in 5 words"}]
)
print(response.content[0].text)

"Hello there, how are you?" 😊


In [7]:
def infer_model(question, current_depth):
    """Decide whether to use Opus or Sonnet based on question complexity."""

    opus_phrases = [
        "explain", "why is", "why are", "why do", "why did",
        "what drives", "what caused", "what factors", "what leads to",
        "how do you explain", "root cause", "interpret", "implications",
        "correlat", "relationship between", "compare and explain",
        "what does this mean", "what can we conclude", "deep dive",
        "assess the impact", "evaluate the effect"
    ]

    sonnet_phrases = [
        "how many", "what is the", "list", "count", "total",
        "average", "sum", "which has the most", "top 5", "top 10",
        "what percentage", "show me"
    ]

    q_lower = question.lower()
    needs_opus = any(phrase in q_lower for phrase in opus_phrases)
    simple_question = any(phrase in q_lower for phrase in sonnet_phrases)

    if needs_opus and not simple_question:
        return "claude-opus-4-6", "interpretive"

    if current_depth == 1 and needs_opus:
        return "claude-opus-4-6", "depth+complexity"

    return "claude-sonnet-4-6", None


def classify_question(question, schema, tables, call_llm):
    """Phase 1: Determine if we can answer the question."""

    prompt = f"""You are a data analyst assistant.

DATA MODEL:
{schema}

Available tables: {list(tables.keys())}

A user asked: {question}

Classify into one of three categories:
- can_answer: the data model is sufficient to answer this question. This includes questions about the data model itself (what tables exist, what columns mean, what use cases are supported).
- cant_answer: requires data not in the model. Reasons: question_domain_mismatch, missing_data, other
- clarifications_needed: too vague or ambiguous
IMPORTANT: If no concrete metric or KPI is specified, and a clear kpi can not be inferred from the context or available metadata, classify as clarifications_needed.
Also classify as clarifications_needed if the question contains assumptions that would make the answer trivially uniform or analytically meaningless (e.g., "profit margin assuming COGS is X% of price" produces identical margins for every category). In such cases, keep the explanation to 1-2 sentences and suggest one alternative question. Do not explain the math or why it's trivial — the user understands, they may have just phrased it differently than intended.

Respond in EXACTLY this format:
CLASSIFICATION: <can_answer|cant_answer|clarifications_needed>
REASON: <REQUIRED. If cant_answer: explain specifically which tables and columns ARE available and which are missing. If clarifications_needed: explain what the user needs to specify. If can_answer: briefly state which tables and columns will be used, or which parts of the data model documentation answer the question.>"""

    text = call_llm(prompt)
    classification = "can_answer"
    reason = ""
    for line in text.split("\n"):
        if line.startswith("CLASSIFICATION:"):
            classification = line.split(":")[1].strip().lower()
        if line.startswith("REASON:"):
            reason = line.split(":", 1)[1].strip()
    return classification, reason


def plan_queries(question, schema, tables, depth, data_processing, call_llm):
    """Phase 2a: Break the question into an array of queries."""

    if depth == 0:
        depth_instruction = """Generate ONLY the minimum queries needed to directly answer the user's question. No additional context or analysis."""
    elif depth == 1:
        depth_instruction = """First, generate queries to directly answer the user's question. Then, assess the user's likely intent and add 2-3 supplementary queries that provide useful context. For example: trends over time, breakdowns by key dimensions, or related metrics that a business user would naturally ask next."""
    else:
        raise ValueError(f"Invalid depth={depth}. Must be 0 or 1.")

    if data_processing == "sql":
        query_lang = "SQL (DuckDB syntax)"
        query_rules = """- Standard SQL compatible with DuckDB
- Use CAST(column AS TIMESTAMP) for date conversions
- Use DuckDB date functions (DATE_PART, DATEDIFF, etc.)
- No semicolons"""
    else:
        query_lang = "Python pandas code"
        query_rules = """- DataFrames are already loaded with the variable names listed
- Do NOT import anything — pandas is 'pd', numpy is 'np'
- Use pd.to_datetime() for date conversions
- Use :.2f for floats, never :d"""

    prompt = f"""You are a data analyst. Break down the user's question into a structured analysis plan.

DATA MODEL:
{schema}

Available tables: {list(tables.keys())}

User question: {question}

{depth_instruction}

You also have direct access to the DATA MODEL documentation above. If the question asks about table descriptions, use cases, column meanings, or data model structure, you can reference that documentation directly in your labels — no query is needed for metadata-only parts. For mixed questions (metadata + data), generate queries for the data parts and note which parts will be answered from documentation.

For each query, provide:
- A short label describing what this query answers
- Whether it's "primary" (directly answers the question) or "supplementary" (adds context)
- The {query_lang} query

Rules for queries:
{query_rules}
- Each query must be independently executable
- Only use tables and columns from the data model
- Do not assume currency, units, or external data not in the model

Respond using XML tags:
<query>
<label>What this query answers</label>
<type>primary</type>
<code>
SELECT ...
</code>
</query>

<query>
<label>What this supplementary query answers</label>
<type>supplementary</type>
<code>
SELECT ...
</code>
</query>

Generate as many query blocks as needed."""

    text = call_llm(prompt)

    import re
    queries = []
    query_blocks = re.findall(r'<query>(.*?)</query>', text, re.DOTALL)
    for block in query_blocks:
        label_match = re.search(r'<label>(.*?)</label>', block, re.DOTALL)
        type_match = re.search(r'<type>(.*?)</type>', block, re.DOTALL)
        code_match = re.search(r'<code>(.*?)</code>', block, re.DOTALL)
        if label_match and code_match:
            queries.append({
                "label": label_match.group(1).strip(),
                "type": type_match.group(1).strip() if type_match else "primary",
                "code": code_match.group(1).strip().rstrip(";"),
                "result": None,
                "error": None
            })

    # Generate execution plan summary
    plan_prompt = f"""Summarize the analysis plan you created in natural language.

User question: {question}
Depth: {"Direct answer only" if depth == 0 else "Direct answer plus contextual analysis"}

Queries planned:
{chr(10).join([f"- [{q['type']}] {q['label']}" for q in queries])}

Write a brief execution plan in this format:
INTENT: <one sentence describing the perceived user intent>
QUERIES: <comma-separated list of what each query answers>
APPROACH: <one sentence on how results will be synthesized>

No markdown, no extra text."""

    execution_plan = call_llm(plan_prompt)

    return queries, execution_plan


def execute_queries(queries, tables, data_processing):
    """Phase 2b: Execute all planned queries."""

    if data_processing == "sql":
        import duckdb
        for q in queries:
            try:
                con = duckdb.connect()
                for name, df in tables.items():
                    con.register(name, df)
                code = q["code"].replace("```", "").strip().rstrip(";")
                result_df = con.execute(code).df()
                q["result"] = result_df.to_string(index=False)
                con.close()
            except Exception as e:
                q["error"] = str(e)
    else:
        for q in queries:
            try:
                local_vars = {name: df.copy() for name, df in tables.items()}
                local_vars["pd"] = pd
                local_vars["np"] = np
                import io, sys
                code = q["code"].replace("```", "").strip()
                code = code.replace(":d}", ":g}")
                output = io.StringIO()
                old_stdout = sys.stdout
                sys.stdout = output
                exec(code, local_vars)
                sys.stdout = old_stdout
                q["result"] = output.getvalue()
            except Exception as e:
                sys.stdout = old_stdout
                q["error"] = str(e)

    return queries


def format_narrative(question, queries, depth, schema, call_llm):
    """Phase 3: Synthesize all query results into a coherent narrative."""

    results_text = ""
    for q in queries:
        results_text += f"\n[{q['type'].upper()}] {q['label']}\n"
        if q["result"]:
            results_text += f"Result:\n{q['result']}\n"
        else:
            results_text += f"Error: {q['error']}\n"

    if depth == 0:
        format_instruction = """Format rules:
- Answer in 1-3 sentences, one per primary query result
- State precise numbers with thousand separators
- If the question has multiple parts, address each part clearly
- Do not add context, interpretation, or follow-up suggestions"""
    else:
        format_instruction = """Format rules:
- Start with a direct answer to the user's question using primary query results, precise numbers with thousand separators
- Address every part of the user's question explicitly
- Then weave in supplementary findings naturally, each as its own short paragraph starting with "If you're interested in [topic],"
- Use precise numbers first, approximations only after
- Skip any queries that failed silently
- End with one suggested follow-up question: "Would you like me to analyze [specific topic] next?"
- The suggested question should be naturally related but different from what was already covered"""

    prompt = f"""Compose a user-friendly analytical answer from these query results.

User question: {question}

DATA MODEL (use for metadata questions about table structure, use cases, column meanings):
{schema}

Query results:
{results_text}

{format_instruction}

General rules:
- You have access to the DATA MODEL documentation above. Use it to answer questions about table structure, use cases, column meanings, or data model capabilities directly — weave metadata and query results together naturally
- When referencing metadata (table descriptions, documented use cases), present them as facts without quoting the documentation verbatim
- Do not assume currency or units unless explicitly in the results
- Do not use adjectives to characterize results as good/bad/impressive/concerning
- Do not show query details, table names, or column names
- Do not repeat raw data verbatim — synthesize into readable prose
- If a query was meant to explain "why" something is the case, provide the explanation using the data
- Do not start sentences with filler phrases like "Based on the query results", "Analysis reveals", "The data shows that"
- It IS acceptable to reference the data source mid-sentence for trust, e.g. "The payments table contains 103,886 records across 5 payment types" or "Credit cards account for 78% of revenue (103,886 payments)"
- If the query results show no meaningful variation (e.g., identical values across all rows), flag this to the user: explain that the result is uniform due to the assumptions in the question, and suggest how to reformulate the question for more useful analysis
- NEVER infer customer intent, motivation, or strategy from purchasing patterns. State what the data shows (e.g. "multi-item orders have higher average freight costs") but do not speculate on why (e.g. "customers strategically bundle to optimize shipping"). Correlation in data does not imply causation or intent.
- When suggesting follow-up questions, never frame them as causal ("what drives", "why do customers", "what causes"). Instead frame as descriptive ("how do multi-item orders differ from single-item orders", "what are the characteristics of multi-item purchasers").

Answer:"""

    return call_llm(prompt)


def analyst_agent(question, schema=SCHEMA, tables=tables, backend="claude",
                  depth=0, data_processing="sql", max_retries=3):
    """3-phase analytics agent: classify → plan & execute → narrate."""

    def call_llm(prompt, model=None):
        if model is None:
            model = current_model
        for attempt in range(max_retries):
            try:
                response = claude_client.messages.create(
                    model=model,
                    max_tokens=4096,
                    messages=[{"role": "user", "content": prompt}]
                )
                return response.content[0].text.strip()
            except Exception as e:
                if "429" in str(e) and attempt < max_retries - 1:
                    print(f"    Rate limited, waiting 15s...")
                    time.sleep(15)
                else:
                    raise e

    current_model, model_reason = infer_model(question, depth)

    # Phase 1: Classify
    try:
        classification, reason = classify_question(question, schema, tables, call_llm)
    except Exception as e:
        return {"stage1": "error", "answer": None, "code": "", "queries": [],
                "execution_plan": None, "narrative": f"Classification failed: {e}", "error": str(e)}

    if classification == "cant_answer":
        answer = f"Can't answer based on the available data. ({reason})"
        return {"stage1": "cant_answer", "answer": answer, "code": "",
                "queries": [], "execution_plan": None, "narrative": answer, "error": None}

    if classification == "clarifications_needed":
        answer = f"Please clarify. {reason}"
        return {"stage1": "clarifications_needed", "answer": answer, "code": "",
                "queries": [], "execution_plan": None, "narrative": answer, "error": None}

    # Phase 2a: Plan queries
    try:
        queries, execution_plan = plan_queries(question, schema, tables, depth, data_processing, call_llm)
    except ValueError as e:
        return {"stage1": "can_answer", "answer": None, "code": "", "queries": [],
                "execution_plan": None, "narrative": str(e), "error": str(e)}
    except Exception as e:
        return {"stage1": "can_answer", "answer": None, "code": "", "queries": [],
                "execution_plan": None, "narrative": f"Query planning failed: {e}", "error": str(e)}

    # Phase 2b: Execute queries
    queries = execute_queries(queries, tables, data_processing)

    # Collect primary results as the raw answer
    primary_results = [q["result"] for q in queries if q["type"] == "primary" and q["result"]]
    raw_answer = "\n".join(primary_results) if primary_results else "No results"

    # Collect all code
    all_code = "\n\n".join([f"-- {q['label']}\n{q['code']}" for q in queries])

    # Phase 3: Narrate
    try:
        narrative = format_narrative(question, queries, depth, schema, call_llm)
    except Exception as e:
        narrative = f"(Narrative failed: {e})"

    return {
        "stage1": classification, "answer": raw_answer, "code": all_code,
        "queries": queries, "execution_plan": execution_plan,
        "narrative": narrative, "error": None
    }

In [8]:
def run_eval(agent_fn, questions, name="agent", md_file=None,
             depth=0, data_processing="sql",
             print_classification=False, print_path=False, print_raw=False,
             print_narrative=False, print_result=True):
    results = []
    passed = 0

    if md_file:
        with open(md_file, "w") as f:
            f.write(f"# Eval: {name} (depth={depth}, processing={data_processing})\n\n")

    for i, q in enumerate(questions):
        print(f"\n  Q{i+1}: {q['question']}")
        r = agent_fn(q["question"], depth=depth, data_processing=data_processing)

        if r["error"]:
            classification = "error"
        elif r["answer"] and any(x in r["answer"].lower() for x in ["can't answer", "cannot answer"]):
            classification = "cant_answer"
        elif r["answer"] and any(x in r["answer"].lower() for x in ["unclear", "clarif", "please specify"]):
            classification = "clarifications_needed"
        elif r["answer"]:
            classification = "can_answer"
        else:
            classification = "no_output"

        success = False
        if r["error"]:
            status = f"ERROR: {r['error'][:120]}"
        elif r["answer"]:
            success = q["validate"](r["answer"])
            if not success and r.get("narrative"):
                success = q["validate"](r["narrative"])
            status = "PASS" if success else "FAIL"
        else:
            status = "NO OUTPUT"

        if print_classification:
            print(f"  Classification: {classification}")

        if print_path:
            if r.get("execution_plan"):
                print(f"  Execution plan: {r['execution_plan']}")
            for j, qr in enumerate(r.get("queries", [])):
                tag = "PRIMARY" if qr["type"] == "primary" else "SUPP"
                print(f"  [{tag}] {qr['label']}")
                print(f"    Query: {qr['code']}")
                if qr["result"]:
                    print(f"    Result: {qr['result']}")
                if qr["error"]:
                    print(f"    Error: {qr['error']}")

        if print_raw and r.get("answer"):
            print(f"  Raw answer: {r['answer'].strip()}")

        if print_narrative and r.get("narrative"):
            print(f"  Narrative: {r['narrative']}")

        if print_result:
            print(f"  → {status}")

        passed += success
        results.append({
            "question": q["question"], "expected": q["expected"],
            "classification": classification, "status": status,
            "code": r["code"], "answer": r["answer"],
            "queries": r.get("queries", []),
            "execution_plan": r.get("execution_plan"),
            "narrative": r.get("narrative"), "error": r["error"]
        })

    print(f"\n{'='*60}")
    print(f"[{name}] Result: {passed}/{len(questions)} passed")
    print(f"{'='*60}")

    if md_file:
        lang = "sql" if data_processing == "sql" else "python"
        with open(md_file, "a") as f:
            for i, res in enumerate(results):
                f.write(f"## Q{i+1}: {res['question']}\n\n")
                f.write(f"**Expected:** {res['expected']}\n\n")
                f.write(f"**Classification:** {res['classification']}\n\n")
                f.write(f"**Status:** {res['status']}\n\n")
                if res.get("execution_plan"):
                    f.write(f"**Execution Plan:**\n{res['execution_plan']}\n\n")
                if res["answer"]:
                    f.write(f"**Raw Answer:**\n```\n{res['answer'].strip()}\n```\n\n")
                for j, qr in enumerate(res.get("queries", [])):
                    tag = "Primary" if qr["type"] == "primary" else "Supplementary"
                    f.write(f"**{tag}: {qr['label']}**\n```{lang}\n{qr['code']}\n```\n")
                    if qr["result"]:
                        f.write(f"```\n{qr['result']}\n```\n\n")
                    if qr["error"]:
                        f.write(f"*Error: {qr['error']}*\n\n")
                if res.get("narrative"):
                    f.write(f"**Final Answer:**\n{res['narrative']}\n\n")
                if res["error"]:
                    f.write(f"**Error:** {res['error']}\n\n")
                f.write("---\n\n")
            f.write(f"\n## Summary: {passed}/{len(questions)} passed\n")

    return {"name": name, "passed": passed, "total": len(questions), "results": results}

In [9]:
def chat():
    """Interactive chat with follow-up resolution and auto-depth."""
    history = []
    print("Analytics Bot (type 'quit' to exit, 'deep' for detailed analysis, 'fast' for quick answers)")
    print("=" * 60)

    current_depth = 0
    auto_depth_set = False

    while True:
        question = input("\nYou: ")

        if question.strip().lower() == "quit":
            print("Session ended.")
            break

        if question.strip().lower() == "deep":
            current_depth = 1
            auto_depth_set = False
            print("  Switched to detailed analysis mode.")
            continue

        manual_override = False

        if question.strip().lower().startswith("fast"):
            current_depth = 0
            auto_depth_set = False
            manual_override = True
            print("  Switched to quick answer mode.")
            remaining = question.strip()[4:].lstrip(":").lstrip()
            if remaining:
                question = remaining
            else:
                continue

        if question.strip().lower().startswith("deep"):
            current_depth = 1
            auto_depth_set = False
            manual_override = True
            print("  Switched to detailed analysis mode.")
            remaining = question.strip()[4:].lstrip(":").lstrip()
            if remaining:
                question = remaining
            else:
                continue

        # Resolve follow-ups into standalone questions
        ack_options = ["Got it", "Sure", "OK", "On it", "Let me check",
               "Right", "Understood", "Will do", "Looking into it", "Comming right up"]

        if history:
            recent = history[-3:]
            context = "\n".join([
                f"User: {h['question']}\nBot: {h['narrative'][:200]}...{h['narrative'][-200:]}"
                for h in recent
            ])

            resolve_response = claude_client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=300,
                messages=[{"role": "user", "content": f"""Given this conversation history:

{context}

The user now says: {question}

If the bot's last message ended with a suggested follow-up question and the user responds with "yes", "sure", "go ahead", "please do", or similar affirmation, rewrite it as that suggested follow-up question.

If this is a follow-up that references the previous exchange (e.g. "break that down by state", "same but for 2017", "why?"), rewrite it as a complete standalone question that includes all necessary context.

When the user asks a short follow-up about a metric (e.g. "avg value?", "how many?", "% of total?"), apply it to the MOST SPECIFIC entity discussed, not a broader set. If the previous exchange identified a single top result, the follow-up refers to that one result, not the entire list.

If it's already a standalone question, return it unchanged."""}]
            )
            resolved_question = resolve_response.content[0].text.strip()

            if resolved_question.lower() != question.lower():
                ack = random.choice([a for a in ack_options if a != history[-1].get("ack", "")] if history else ack_options)
                short = resolved_question[:80].rstrip("?").lower()
                if len(resolved_question) > 80:
                    short += "..."
                print(f"  {ack} — {short}")
        else:
            resolved_question = question

        if not manual_override:
            # Infer depth from question phrasing
            deep_phrases = ["analyze", "analyse", "explain", "assess", "evaluate",
                            "propose", "investigate", "deep dive", "in depth",
                            "broadly", "comprehensive", "thorough", "explore",
                            "why is", "why are", "why do", "what drives",
                            "what factors", "root cause", "understand"]

            inferred_depth = current_depth
            depth_reason = None

            # Check for deep phrases in the original question
            q_lower = question.lower()
            if current_depth == 0 and any(phrase in q_lower for phrase in deep_phrases):
                inferred_depth = 1
                depth_reason = "phrasing"

            # Auto-escalate after 3rd follow-up
            follow_up_count = len(history)
            if current_depth == 0 and not auto_depth_set and follow_up_count >= 3:
                inferred_depth = 1
                depth_reason = "conversation depth"

            # Notify user of depth change
            if inferred_depth != current_depth and not auto_depth_set:
                if depth_reason == "phrasing":
                    print(f"  Switching to detailed analysis mode based on your question.")
                else:
                    print(f"  You seem to be digging deeper — switching to detailed analysis mode.")
                print(f"  (Type 'fast' to switch back to quick answers)")
                current_depth = inferred_depth
                auto_depth_set = True

        # Run the agent
        model_name, model_reason = infer_model(question, current_depth)
        if model_reason and "opus" in model_name:
            print(f"  Using advanced reasoning for deeper analysis (Claude Opus 4.6 model).")
        elif hasattr(chat, '_last_model') and "opus" in getattr(chat, '_last_model', '') and "sonnet" in model_name:
            print(f"  Back to Sonnet.")
        chat._last_model = model_name

        r = analyst_agent(resolved_question, depth=current_depth)

        history.append({
            "question": question,
            "resolved": resolved_question,
            "narrative": r["narrative"],
            "depth": current_depth
        })

        print(f"\nBot: {r['narrative']}")

chat()

Analytics Bot (type 'quit' to exit, 'deep' for detailed analysis, 'fast' for quick answers)
  Switching to detailed analysis mode based on your question.
  (Type 'fast' to switch back to quick answers)
  Upgrading to Opus for deeper analysis.

Bot: ## How Freight Costs Influence Order Value Fluctuations Between Categories

Freight costs account for vastly different shares of total item value depending on the product category, ranging from as little as **4.23%** for PCs to as much as **26.82%** for Christmas articles (*artigos_de_natal*). This spread means that freight is a minor rounding error in some categories but a significant component of what customers actually pay in others.

**Categories where freight dominates order value fluctuations** include Christmas articles (26.82% freight share), signage & security (23.23%), food & beverages (22.89%), and electronics (22.66%). These tend to be lower-priced items — averaging R$54–R$108 in product price — where even a modest R$16–R$33 frei

In [ ]:
# Sandbox

# for d in [0, 1]:
#     r = analyst_agent("How many unique customers are there? Of these, how many in our top-selling city (Gross Revenue)? Was this city always our top-selling one? If not, since when?", depth=d)
#     print("=== Execution Plan ===")
#     print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [1]:
#     r = analyst_agent("Tell me about our company. What are we selling? Typical products, prices, locations.", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [1]:
#     r = analyst_agent("In which product categories do we struggle (Gross Revenue, Customer Satrisfaction, other meaningful KPIs)? Try to explain potential underlying reasons.", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [1]:
#     r = analyst_agent("Analyze the correlation between product weight, shipping costs, and customer satisfaction to identify logistics optimization opportunities.", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [0]:
#     r = analyst_agent("Which supplier demonstrates highest rate of customer repeat-sales (multiple purchases per customer, from the same supplier). Exclude cases for which number of unique customers buying from the supplier is less than 10. Use net revenue (excl. cancelled orders)?", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [0]:
#     r = analyst_agent("What is my cost structure? Do I pay commissions to suppliers, or is it cogs only? Something else?", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [0]:
#     r = analyst_agent("Do you see problematic geographic areas, where gross sales might be inhibited by lack of sufficient payment method options?", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])

# for d in [0]:
#     r = analyst_agent("What are the listed use cases and main kpis for the payments table? ", depth=d)
#     # print("=== Execution Plan ===")
#     # print(r["execution_plan"])
#     print("\n=== Narrative ===")
#     print(r["narrative"])


=== Execution Plan ===
INTENT: The user wants to know the total unique customer count, identify the top revenue-generating city and its customer count, and determine when that city first became the top seller.

QUERIES: Count all unique customers across the dataset, identify the city with highest gross revenue and count its unique customers, analyze monthly revenue trends by city to find when the top city first achieved that position.

APPROACH: I will execute the three queries sequentially and combine the results to provide the total customer count, top city details, and historical timeline of when it became the leading revenue generator.

=== Narrative ===
There are 96,096 unique customers in the dataset. Of these, 14,761 customers are located in São Paulo, which is the top-selling city with 2,150,668.09 in gross revenue. São Paulo became the top-selling city in January 2017 and maintained that position consistently through September 2018, though it was not the top city in the earlie

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


=== Narrative ===
The Brazilian marketplace data reveals several product categories struggling across key performance metrics. The most problematic categories show significant challenges in customer satisfaction, delivery performance, and market dynamics.

**Fashion and furniture categories face the biggest customer satisfaction problems.** Fashion menswear performs worst with an average satisfaction score of 3.76 and 22.6% of orders receiving low ratings (scores of 1-3). Audio equipment also struggles significantly with 21.6% low satisfaction orders and an average score of 3.83. Furniture categories consistently underperform: office furniture (3.52 average score, 21.9% low satisfaction), living room furniture (3.94 average score), and home comfort items (3.85 average score).

**These struggling categories share common underlying issues.** Office furniture shows the most severe delivery problems with an average of 20.8 days to delivery compared to the marketplace average of around 12 

KeyboardInterrupt: 

In [ ]:
easy_test_questions = [
    {
        "question": "What is the average item price across all orders?",
        "validate": lambda ans: any(x in ans for x in ["120", "121", "160", "161"]),
        "expected": "~120 per item or ~160 per order (both valid interpretations)"
    },
    {
        "question": "Which city has the most customers?",
        "validate": lambda ans: "sao paulo" in ans.lower(),
        "expected": "Sao Paulo"
    },
    {
        "question": "What is the average review score?",
        "validate": lambda ans: "4.0" in ans or "4.1" in ans,
        "expected": "~4.0-4.1 average score"
    },
    {
        "question": "What percentage of orders paid by credit card were paid by credit card?",
        "validate": lambda ans: any(x in ans.lower() for x in ["clarif", "please", "trivial", "100%", "by definition", "tautolog", "always"]),
        "expected": "Should flag as trivially meaningless — answer is always 100% by definition"
    },
    {
        "question": "How many orders were delivered late (delivered after estimated delivery date)?",
        "validate": lambda ans: any(char.isdigit() for char in ans),
        "expected": "A numeric count of late deliveries"
    },
    {
        "question": "What is the total revenue by payment type?",
        "validate": lambda ans: "credit_card" in ans.lower(),
        "expected": "Breakdown including credit_card as top type"
    },
    {
        "question": "Which seller has the most orders?",
        "validate": lambda ans: len(ans.strip()) > 0,
        "expected": "A seller ID"
    },
    {
        "question": "What is the month with the highest number of orders?",
        "validate": lambda ans: any(char.isdigit() for char in ans),
        "expected": "A month (number or name)"
    },
    {
        "question": "What is the average freight cost per item across all ecommerce companies worldwide?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "not available", "only", "this data", "one company", "single", "does not contain", "doesn't contain", "unable", "no data"]),
        "expected": "Should decline: data only covers one company, not all ecommerce worldwide"
    },
    {
        "question": "What's the best ever?",
        "validate": lambda ans: any(x in ans.lower() for x in ["clarif", "unclear", "specify", "which", "what do you mean", "ambiguous", "please provide", "what metric", "define"]),
        "expected": "Should ask for clarification: best by which metric? which entity?"
    },
]

In [ ]:
hard_test_questions = [
    {
        "question": "What is the average delivery time in days for each customer state, only for delivered orders?",
        "validate": lambda ans: "SP" in ans and any(char.isdigit() for char in ans),
        "expected": "A breakdown by state with SP included, showing average days"
    },
    {
        "question": "Which product category has the highest percentage of 1-star reviews?",
        "validate": lambda ans: any(char.isalpha() for char in ans) and ("%" in ans or "." in ans),
        "expected": "A category name with a percentage or ratio"
    },
    {
        "question": "What is the repeat purchase rate? That is, what percentage of unique customers placed more than one order?",
        "validate": lambda ans: "%" in ans or "." in ans,
        "expected": "A low percentage (~3%) since most customers order once"
    },
    {
        "question": "For orders paid by credit card, what is the average number of installments and how does average order value differ by installment count?",
        "validate": lambda ans: "installment" in ans.lower() or any(str(i) in ans for i in range(1, 13)),
        "expected": "A table or breakdown showing installment counts with corresponding avg order values"
    },
    {
        "question": "Which sellers have an average review score below 2.0 and more than 10 orders? List them with their city and order count.",
        "validate": lambda ans: "seller" in ans.lower() or len(ans.strip()) > 50,
        "expected": "A list of seller IDs with city and order count, or a statement that none qualify"
    },
    {
        "question": "What is the month-over-month order growth rate for 2017?",
        "validate": lambda ans: "2017" in ans or ("%" in ans and any(char.isdigit() for char in ans)),
        "expected": "Monthly growth rates as percentages for 2017"
    },
    {
        "question": "Is there a correlation between product weight and freight cost? What is the Pearson correlation coefficient?",
        "validate": lambda ans: "0." in ans or "corr" in ans.lower(),
        "expected": "A correlation coefficient between 0 and 1"
    },
    {
        "question": "What are the top 5 seller-customer state pairs by order volume? For example, seller in SP shipping to RJ.",
        "validate": lambda ans: "SP" in ans and len(ans.strip()) > 30,
        "expected": "Pairs like SP->SP, SP->RJ etc. with counts"
    },
    {
        "question": "What is the average profit margin per category, assuming COGS is 60% of the item price?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "not available", "assumption", "no cost", "no cogs"]) or ("%" in ans and any(char.isalpha() for char in ans)),
        "expected": "Declines (no real COGS data), the question leads to meaningless analysis."
    },
    {
        "question": "Compare the delivery performance of the top 3 carriers by volume: what percentage of their orders arrived late?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no carrier", "not available", "does not contain"]),
        "expected": "Should decline: the dataset has no carrier information"
    },
]

In [ ]:
misleading_test_questions = [
    # Judgment calls - needs clarification
    {
        "question": "Who is our best customer?",
        "validate": lambda ans: any(x in ans.lower() for x in ["clarif", "unclear", "specify", "which", "metric", "define", "what do you mean", "criteria"]),
        "expected": "clarifications_needed: best by what? revenue, order count, review score?"
    },
    {
        "question": "Which sellers should we drop from the platform?",
        "validate": lambda ans: any(x in ans.lower() for x in ["clarif", "unclear", "specify", "criteria", "define", "what", "threshold", "based on"]),
        "expected": "clarifications_needed: drop based on what criteria? low reviews, late delivery, low volume?"
    },
    {
        "question": "Is our business doing well?",
        "validate": lambda ans: any(x in ans.lower() for x in ["clarif", "unclear", "specify", "define", "what", "metric", "compared"]),
        "expected": "clarifications_needed: well compared to what? which KPI?"
    },

    # Speculation with partial data - has local data but needs external reference
    {
        "question": "How does the average order value on our platform compare to the average ecommerce order value in Brazil?",
        "validate": lambda ans: any(x in ans.lower() for x in ["external", "outside", "can't answer", "cannot answer", "not available", "only", "no data", "benchmark", "don't have"]),
        "expected": "Can calculate our AOV but should flag that Brazil-wide benchmark is not in the dataset"
    },
    {
        "question": "Are our customers in São Paulo wealthier than the national average?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no income", "no wealth", "not available", "no data", "demographic", "missing"]),
        "expected": "cant_answer: no income or wealth data in the dataset at all"
    },

    # Wrong table / wrong source
    {
        "question": "Based on the customers table, which customer segment is most profitable?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no segment", "no profit", "not available", "no cost", "missing", "does not contain"]),
        "expected": "cant_answer: customers table has no segment or profitability fields"
    },
    {
        "question": "What is the return rate by product category?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no return", "not available", "missing", "does not contain"]),
        "expected": "cant_answer: no returns data in the dataset"
    },

    # Misleading / sounds answerable but isn't
    {
        "question": "What is the conversion rate from cart to purchase by product category?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no cart", "no funnel", "not available", "missing", "no browsing", "no event"]),
        "expected": "cant_answer: no cart or browsing funnel data, only completed orders"
    },
    {
        "question": "What is the customer lifetime value for the top 10 customers?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "clarif", "lifetime", "define", "assumption", "limited"]) or any(char.isdigit() for char in ans),
        "expected": "Either clarifications_needed (LTV definition varies) or attempts an answer with caveats (limited repeat purchase data)"
    },

    # Completely off-domain
    {
        "question": "What was the weather in São Paulo on the day with the most orders?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no weather", "not available", "external", "outside", "missing"]),
        "expected": "cant_answer: no weather data in the dataset"
    },
]

In [ ]:
multistage_test_questions = [
    # Stage 1+2: Product size classification → sales comparison
    {
        "question": "First, classify products into 'small', 'medium', and 'bulky' based on the 33rd and 66th percentiles of product_weight_g. Then show total revenue and average review score for each size class.",
        "validate": lambda ans: all(x in ans.lower() for x in ["small", "medium", "bulky"]) and any(char.isdigit() for char in ans),
        "expected": "Three size classes with revenue and avg review score for each"
    },

    # Stage 1+2+3: Best month → top categories → consistency
    {
        "question": "Find the month with the highest total revenue. Then identify the top 3 product categories in that month. Finally, show how those same 3 categories performed in every other month of the same year — were they consistently top sellers or just spiking in that one month?",
        "validate": lambda ans: any(char.isdigit() for char in ans) and any(char.isalpha() for char in ans) and len(ans.strip()) > 100,
        "expected": "A month identified, top 3 categories named, then a monthly breakdown of those categories across the year"
    },

    # Stage 1+2: Customer segmentation → behavior comparison
    {
        "question": "Segment customers into 'one-time' and 'repeat' based on whether customer_unique_id appears in more than one order. Then compare these two segments on: average order value, average review score, and average delivery time in days.",
        "validate": lambda ans: all(x in ans.lower() for x in ["one-time", "repeat"]) or ("one" in ans.lower() and "repeat" in ans.lower()),
        "expected": "Two segments with three metrics compared side by side"
    },
    # Stage 1+2: Seller tier → late delivery analysis
    {
        "question": "Rank sellers into 'high volume' (top 10% by order count), 'medium volume' (next 40%), and 'low volume' (bottom 50%). Then calculate the percentage of late deliveries for each seller tier.",
        "validate": lambda ans: ("high" in ans.lower() or "top" in ans.lower()) and ("%" in ans or "late" in ans.lower()),
        "expected": "Three seller tiers with late delivery percentages"
    },

    # Stage 1+2: Geographic distance proxy → delivery impact
    {
        "question": "Classify orders as 'same state' or 'cross state' based on whether the seller state matches the customer state. Then compare average delivery time, average freight cost, and average review score between the two groups.",
        "validate": lambda ans: ("same" in ans.lower() and "cross" in ans.lower()) and any(char.isdigit() for char in ans),
        "expected": "Two groups with delivery time, freight cost, and review score compared"
    },
]

In [ ]:
realistic_test_questions = [
    {
        "question": "How many unique customers are there? Of these, how many in our top-selling city (Gross Revenue)? Was this city always our top-selling one? If not, since when?",
        "validate": lambda ans: "96096" in ans.replace(",", "").replace(" ", "") or "96,096" in ans,
        "expected": "96,096 unique customers, São Paulo as top city with customer count, and historical city ranking"
    },
    {
        "question": "Tell me about our company. What are we selling? Typical products, prices, locations.",
        "validate": lambda ans: any(char.isdigit() for char in ans) and any(x in ans.lower() for x in ["categor", "product", "price", "city", "state"]),
        "expected": "Overview of product categories, price ranges, and geographic footprint"
    },
    {
        "question": "In which product categories do we struggle (Gross Revenue, Customer Satisfaction, other meaningful KPIs)? Try to explain potential underlying reasons.",
        "validate": lambda ans: any(char.isalpha() for char in ans) and len(ans.strip()) > 100,
        "expected": "Low-performing categories identified with revenue, review scores, and possible explanations"
    },
    {
        "question": "Analyze the correlation between product weight, shipping costs, and customer satisfaction to identify logistics optimization opportunities.",
        "validate": lambda ans: ("corr" in ans.lower() or "0." in ans) and any(x in ans.lower() for x in ["weight", "freight", "review", "score", "shipping"]),
        "expected": "Correlation coefficients between weight/freight/reviews with optimization insights"
    },
    {
        "question": "Which supplier demonstrates highest rate of customer repeat-sales (multiple purchases per customer, from the same supplier). Exclude cases for which number of unique customers buying from the supplier is less than 10. Use net revenue (excl. cancelled orders)?",
        "validate": lambda ans: any(char.isalnum() for char in ans) and ("seller" in ans.lower() or len(ans.strip()) > 30),
        "expected": "A seller ID with repeat purchase rate, filtered by minimum 10 customers, using non-cancelled orders"
    },
    {
        "question": "What is my cost structure? Do I pay commissions to suppliers, or is it cogs only? Something else?",
        "validate": lambda ans: any(x in ans.lower() for x in ["can't answer", "cannot answer", "no cost", "not available", "missing"]),
        "expected": "cant_answer: no cost structure, COGS, or commission data in the dataset"
    },
    {
        "question": "What are the listed use cases and main kpis for the payments table?",
        "validate": lambda ans: any(x in ans.lower() for x in ["payment", "revenue", "installment", "credit", "method"]),
        "expected": "Use cases and KPIs from the data model metadata, optionally enriched with actual data statistics"
    },
    {
        "question": "Do you see problematic geographic areas, where gross sales might be inhibited by lack of sufficient payment method options?",
        "validate": lambda ans: any(x in ans.lower() for x in ["state", "credit", "boleto", "payment", "geographic", "region"]) and any(char.isdigit() for char in ans),
        "expected": "Geographic breakdown of payment method availability with identification of areas with limited options"
    },

]

In [ ]:
# Execute validation tests

questions = easy_test_questions
print("=== Easy questions ===")
easy_q_results = run_eval(analyst_agent, questions, "Easy questions", md_file="/content/easy_q_results.md", depth=0, data_processing="sql", print_narrative=True)


questions = hard_test_questions
print("=== Hard questions ===")
hard_q_results = run_eval(analyst_agent, questions, "Hard questions", md_file="/content/hard_q_results.md", depth=0, data_processing="sql", print_narrative=True)

questions = misleading_test_questions
print("=== Misleading questions ===")
misleading_q_results = run_eval(analyst_agent, questions, "Misleading questions", md_file="/content/misleading_q_results.md", depth=0, data_processing="sql", print_narrative=True)

questions = multistage_test_questions
print("=== Multi-stage questions ===")
multistage_q_results = run_eval(analyst_agent, questions, "Multi-stage questions", md_file="/content/multistage_q_results.md", depth=0, data_processing="sql", print_narrative=True)

questions = realistic_test_questions
print("=== Realistic business questions ===")
realistic_q_results = run_eval(analyst_agent, questions, "Realistic business questions", md_file="/content/realistic_q_results.md", depth=0, data_processing="sql", print_narrative=True)



=== Easy questions ===

  Q1: What is the average item price across all orders?
  Narrative: The average item price across all orders is 120.65 BRL (Brazilian Real).
  → PASS

  Q2: Which city has the most customers?
  Narrative: São Paulo has the most customers with 14,984 unique customers.
  → PASS

  Q3: What is the average review score?
  Narrative: The average review score across all reviews is 4.09 out of 5, indicating generally high customer satisfaction on the Brazilian marketplace.
  → PASS

  Q4: What percentage of orders paid by credit card were paid by credit card?
  Narrative: The question is unclear. The question asks "What percentage of orders paid by credit card were paid by credit card?" which is logically circular and would always yield 100%. You likely meant to ask something like "What percentage of all orders were paid by credit card?" or "What percentage of credit card orders used installments?"
  → PASS

  Q5: How many orders were delivered late (delivered after e

In [ ]:
# Export md file

for f in glob.glob("/content/*.md"):
    print(f"Downloading: {f}")
    files.download(f)

Downloading: /content/realistic_q_results.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/misleading_q_results.md


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: /content/easy_q_results.md


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>